# NBA Game Data — EDA

Read raw game data from S3 and explore.

In [ ]:
import json
import boto3
import pandas as pd

S3_BUCKET = "prediction-markets-data"
s3 = boto3.client("s3")

In [ ]:
# List what's in the bucket
resp = s3.list_objects_v2(Bucket=S3_BUCKET, Prefix="nba/")
for obj in resp.get("Contents", []):
    print(f"{obj['Key']}  ({obj['Size'] / 1024:.0f} KB)")

In [ ]:
# Load a season from S3
SEASON = "2024-25"

obj = s3.get_object(Bucket=S3_BUCKET, Key=f"nba/games/season_{SEASON}.json")
raw = json.loads(obj["Body"].read())
df = pd.DataFrame(raw)
print(f"{len(df)} rows, {df['GAME_ID'].nunique()} games")
df.head()

In [ ]:
df.dtypes

In [ ]:
df.describe()

In [ ]:
# Win/loss distribution by team
record = df.groupby("TEAM_ABBREVIATION")["WL"].value_counts().unstack(fill_value=0)
record["win_pct"] = record["W"] / (record["W"] + record["L"])
record.sort_values("win_pct", ascending=False)

In [ ]:
# Points per game by team
ppg = df.groupby("TEAM_ABBREVIATION")["PTS"].mean().sort_values(ascending=False)
ppg.plot.bar(title="Points Per Game", figsize=(14, 4))

In [ ]:
# Scoring distribution
df["PTS"].hist(bins=30, figsize=(10, 4))

In [ ]:
# Home vs away win rates
df["is_home"] = df["MATCHUP"].str.contains("vs.")
home_wr = df.groupby("is_home")["WL"].value_counts(normalize=True).unstack()
home_wr